# Gate 1: generate candidates with a second model (GPU step)

This notebook only **generates** code. Labelling, pool building and all evaluation run on the laptop (CPU).

**Settings:** Accelerator **GPU T4 x1**, Internet **on**. For a private repo, add a Kaggle secret named `GITHUB_TOKEN` (*Add-ons → Secrets*).

**Then:** *Save Version → Save & Run All*. When it finishes, download `gen_<model>.zip` from the Output tab.

Prompt, seeds and decoding settings are the same as the Baseline 1 run (prompt hash `ebbe8bceae71`).

**New datasets (taxi, titanic):** run it **twice**, once with `MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"` and once with `"Qwen/Qwen2.5-1.5B-Instruct"`, so the new tasks get the same two generators as pool v2.

In [1]:
# ---------------- Configuration ----------------
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
N_SAMPLES = 5
TEMPERATURE = 0.7
LIMIT = None            # e.g. 3 for a quick dry run
DATASETS = ["taxi", "titanic"]   # None = all tasks; pools v1/v2 already cover adult and airbnb
TAG = MODEL_ID.split("/")[-1].lower().replace(".", "")
OUT = f"/kaggle/working/gen_{TAG}" + ("_" + "_".join(DATASETS) if DATASETS else "")
print(TAG, OUT)

qwen25-3b-instruct /kaggle/working/gen_qwen25-3b-instruct_taxi_titanic


In [2]:
# ---------------- Get the code (no pip install: the kernel's torch/transformers are used as is) ----------------
import os, shutil, subprocess
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)        # always a fresh clone of the latest push
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
print(subprocess.run(["git", "log", "-1", "--format=%H %s (%cd)"], capture_output=True, text=True).stdout)

NEEDED = ["src/igt/__init__.py", "src/igt/agents/__init__.py", "src/igt/agents/code_generator.py",
          "eval/generate.py", "eval/prompts/schema_b1-v1.json", "eval/tasks"]
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"These files are not in the GitHub repo (copy them from the workspace and push): {missing}"
print("all needed files present")

no GITHUB_TOKEN secret; cloning without a token (public repo)


Cloning into '/kaggle/working/repo'...


228c593da8abb330fe2978f891e5c0719effda9b updated tasks (Fri Oct 30 14:43:48 2026 +0530)

all needed files present


In [3]:
# ---------------- Check the GPU and the prompt ----------------
import sys, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable the GPU accelerator")
sys.path.insert(0, "src")
from igt.agents.code_generator import PROMPT_HASH
assert PROMPT_HASH == "ebbe8bceae71", PROMPT_HASH
print("prompt hash OK:", PROMPT_HASH)

GPU: Tesla T4
prompt hash OK: ebbe8bceae71


In [4]:
# ---------------- Generate (resumable) ----------------
cmd = ["python", "eval/generate.py", "--model", MODEL_ID, "--out", OUT,
       "--n-samples", str(N_SAMPLES), "--temperature", str(TEMPERATURE)]
if DATASETS:
    cmd += ["--datasets", *DATASETS]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-3000:])
print(p.stderr[-3000:])
assert p.returncode == 0

greedy 8/60 (9s)
greedy 16/60 (20s)
greedy 24/60 (31s)
greedy 32/60 (37s)
greedy 40/60 (42s)
greedy 48/60 (48s)
greedy 56/60 (52s)
greedy 60/60 (54s)
sampled 1/60 (60s)
sampled 2/60 (69s)
sampled 3/60 (75s)
sampled 4/60 (80s)
sampled 5/60 (87s)
sampled 6/60 (91s)
sampled 7/60 (101s)
sampled 8/60 (110s)
sampled 9/60 (115s)
sampled 10/60 (121s)
sampled 11/60 (129s)
sampled 12/60 (135s)
sampled 13/60 (148s)
sampled 14/60 (160s)
sampled 15/60 (172s)
sampled 16/60 (184s)
sampled 17/60 (196s)
sampled 18/60 (206s)
sampled 19/60 (211s)
sampled 20/60 (216s)
sampled 21/60 (221s)
sampled 22/60 (235s)
sampled 23/60 (249s)
sampled 24/60 (255s)
sampled 25/60 (262s)
sampled 26/60 (268s)
sampled 27/60 (274s)
sampled 28/60 (281s)
sampled 29/60 (289s)
sampled 30/60 (295s)
sampled 31/60 (298s)
sampled 32/60 (301s)
sampled 33/60 (304s)
sampled 34/60 (308s)
sampled 35/60 (314s)
sampled 36/60 (317s)
sampled 37/60 (320s)
sampled 38/60 (324s)
sampled 39/60 (328s)
sampled 40/60 (332s)
sampled 41/60 (335s)
samp

In [5]:
# ---------------- Quick look and zip ----------------
import json, shutil, collections
G = [json.loads(l) for l in open(f"{OUT}/generations.jsonl")]
print("generations:", len(G), "| truncated:", sum(g["truncated"] for g in G))
print("per split:", collections.Counter(g["split"] for g in G))
print(G[0]["raw_output"][:500])
open(f"{OUT}/git_commit.txt", "w").write(subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout)
print("saved:", shutil.make_archive(OUT, "zip", OUT))
os.chdir("/kaggle/working")
shutil.rmtree("/kaggle/working/repo", ignore_errors=True)   # keep only the outputs

generations: 360 | truncated: 0
per split: Counter({'test': 300, 'dev': 60})
```python
def transform(df):
    df['fare_plus_tip'] = df['fare_amount'] + df['tip_amount']
    return df
```
saved: /kaggle/working/gen_qwen25-3b-instruct_taxi_titanic.zip
